# 17 — M3 sur la table du pipeline (la génération 2 : mince, sur `tools/`)

**Ce que ce notebook est.** La ré-exécution de la lignée titre sur la **table courante**
(`data/clean/transactions_backtest_2014_2026.csv`, produite par `common/backtest_clean.py`,
step 7 du pipeline — corrections détaillées dans `NOTE_DIFF_TABLE_CLEAN.md`, branche `presentation`),
via le paquet **`tools/`** — dont la fidélité au notebook 16 est prouvée par `tools/test_ancres.py`
(les ancres v1 sont reproduites sur la table v1 archivée).

**Ce que ce notebook n'est pas.** Il ne remplace pas le notebook 16 : le 16 reste la preuve
complète (protocole, validations, §20) sur la table v1 du 04/07 — celle sur laquelle `FICHE_M3`
et la carte restent adossées. Ici : **les mêmes mesures, la table corrigée, l'écart dit** —
la vague de re-certification du Temps 2, en un seul endroit.

Colonnes des tableaux : **v1 (publié)** = la valeur de `FICHE_M3` · **courante** = recalculée ici
· **Δ** = courante − v1.

In [1]:
import sys; sys.path.insert(0, "..")   # tools/ vit a la racine du 02
import numpy as np, pandas as pd
from tools.donnees import charger
from tools import moteur, mesure, etf, poches

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
D = charger()                                        # ← LA TABLE COURANTE (celle du pipeline)
print(f"flux courant : {len(D.df):,} opérations (v1 : 113 369 — Δ {len(D.df)-113_369:+,})")

flux 113,645 opérations · 350 membres · 2,748 titres · calendrier 3,645 j · 149 coupes dès 2014-02-28
flux courant : 113,645 opérations (v1 : 113 369 — Δ +276)


## §1 — M3 sur les titres : les trois portefeuilles

Protocole inchangé (achats seuls, 3 ans à δ, score $B_i m_i$, top-150, plafond 10 %,
close J+1, parts figées) — seule la table change.

In [2]:
V1 = {  # les valeurs PUBLIÉES (FICHE_M3, table v1 du 04/07)
  "NANC":   {"excès": 3.40, "t": 1.61, "α₁": 1.66, "β": 1.080, "NAV": 661.57, "ans": "9/13"},
  "GOP":    {"excès": 1.24, "t": 1.61, "α₁": 1.09, "β": 1.004, "NAV": 573.57, "ans": "9/13"},
  "UNIQUE": {"excès": 2.51, "t": 1.93, "α₁": 1.48, "β": 1.05,  "NAV": 629.09, "ans": "8/13"}}

NAV, B = {}, {}
for parti, nom in [("Democrat", "NANC"), ("Republican", "GOP"), (None, "UNIQUE")]:
    NAV[nom], _ = moteur.run_livre(D, moteur.cibles(D, parti))
    B[nom] = mesure.bilan_ic(D, NAV[nom], f"M3 · {nom} (courante)")

L = []
for nom in ["NANC", "GOP", "UNIQUE"]:
    b, v = B[nom], V1[nom]
    L.append({"run": nom,
              "excès v1": v["excès"], "excès courante": round(b["excès %/an"], 2),
              "Δ excès": round(b["excès %/an"] - v["excès"], 2),
              "t v1": v["t"], "t courante": round(b["t"], 2),
              "α₁ v1": v["α₁"], "α₁ courante": round(b["α %/an"], 2),
              "β v1": v["β"], "β courante": round(b["β"], 3),
              "NAV v1": v["NAV"], "NAV courante": round(b["NAV"], 2),
              "ans v1": v["ans"], "ans courante": b["ann. gagnantes"]})
COMP = pd.DataFrame(L).set_index("run")
display(COMP)

,excès v1,excès courante,Δ excès,t v1,t courante,α₁ v1,α₁ courante,β v1,β courante,NAV v1,NAV courante,ans v1,ans courante
run,,,,,,,,,,,,,
NANC,3.40,3.42,0.02,1.61,1.61,1.66,1.67,1.080,1.080,661.57,662.37,9/13,9/13
GOP,1.24,1.21,-0.03,1.61,1.59,1.09,1.10,1.004,1.003,573.57,572.10,9/13,9/13
UNIQUE,2.51,2.51,-0.00,1.93,1.92,1.48,1.48,1.050,1.053,629.09,628.75,8/13,8/13


## §2 — D'où vient l'excès : la pondération, à univers identique (côté démocrate)

In [3]:
V1_MODES = {"équipondéré": -1.38, "élus seuls": -0.65, "dollars purs": 1.43, "M3 (score)": 3.40}
L = []
for lab, mode in [("équipondéré", "equal"), ("élus seuls", "elus"),
                  ("dollars purs", "dollars"), ("M3 (score)", "score")]:
    V, _ = moteur.run_livre(D, moteur.cibles(D, "Democrat", mode=mode))
    b = mesure.bilan(D, V, f"NANC · {lab} (courante)")
    L.append({"pondération": lab, "excès v1": V1_MODES[lab],
              "excès courante": round(b["excès %/an"], 2),
              "Δ": round(b["excès %/an"] - V1_MODES[lab], 2), "NAV": round(b["NAV"], 1)})
display(pd.DataFrame(L).set_index("pondération"))
x = {r["pondération"]: r["excès courante"] for r in L}
assert x["M3 (score)"] > max(x["dollars purs"], x["élus seuls"], x["équipondéré"]), \
    "le produit ne dépasse plus ses composantes — à investiguer"
print("✅ le produit dépasse toujours ses deux composantes sur la table courante")

,excès v1,excès courante,Δ,NAV
pondération,,,,
équipondéré,-1.38,-1.39,-0.01,429.3
élus seuls,-0.65,-0.65,0.00,463.6
dollars purs,1.43,1.43,0.00,554.0
M3 (score),3.40,3.42,0.02,662.4


✅ le produit dépasse toujours ses deux composantes sur la table courante


## §3 — L'α à quatre facteurs, et le contrôle SPY

In [4]:
V1_A4 = {"NANC": (2.09, 2.08), "GOP": (1.65, 1.79)}
L = []
for nom in ["NANC", "GOP"]:
    f = mesure.facteurs(D, NAV[nom], f"M3 · {nom}")
    L.append({"run": nom, "α₄ v1": V1_A4[nom][0], "α₄ courante": round(f["α %/an"], 2),
              "t v1": V1_A4[nom][1], "t courante": round(f["t_α"], 2),
              "SMB": round(f["SMB"], 3), "HML": round(f["HML"], 3), "Mom": round(f["Mom"], 3)})
display(pd.DataFrame(L).set_index("run"))
nav_spy = (1 + D.r_spy).cumprod() * 100
ctl = mesure.facteurs(D, nav_spy.dropna(), "SPY (contrôle)")
assert abs(ctl["Mkt-RF"] - 1) < 0.05 and abs(ctl["α %/an"]) < 2.0
print(f"✅ contrôle : le SPY au même moulin — β {ctl['Mkt-RF']:.3f} ≈ 1 · α {ctl['α %/an']:+.2f} %/an ≈ 0")

,α₄ v1,α₄ courante,t v1,t courante,SMB,HML,Mom
run,,,,,,,
NANC,2.09,2.10,2.08,2.09,-0.131,-0.146,-0.077
GOP,1.65,1.65,1.79,1.79,-0.082,0.050,-0.073


✅ contrôle : le SPY au même moulin — β 0.980 ≈ 1 · α +0.04 %/an ≈ 0


## §4 — Le net de frais, le filtre θ, le netting

In [5]:
L = []
for parti, nom, v1net in [("Democrat", "NANC", 3.23), ("Republican", "GOP", 1.06)]:
    WP = moteur.cibles(D, parti)
    net, _ = moteur.run_livre(D, WP, cout_bps=10.0)
    b = mesure.bilan(D, net, f"M3 · {nom} net")
    _, turns = moteur.run_livre(D, WP)
    L.append({"run": nom, "excès net v1": v1net, "excès net courante": round(b["excès %/an"], 2),
              "NAV nette": round(b["NAV"], 1), "rotation %/mois (méd.)": round(100*np.median(turns), 1)})
display(pd.DataFrame(L).set_index("run"))

,excès net v1,excès net courante,NAV nette,rotation %/mois (méd.)
run,,,,
NANC,3.23,3.25,649.5,4.9
GOP,1.06,1.03,560.2,5.3


In [6]:
V1_THETA = {"NANC": {10: 4.71, 20: 4.62, 50: 5.14, 100: 4.75}, "GOP": {10: 3.02, 20: 2.14, 50: 1.44, 100: 1.64}}
L = []
for parti, nom in [("Democrat", "NANC"), ("Republican", "GOP")]:
    for th in [10, 20, 50, 100]:
        V, _ = moteur.run_livre(D, moteur.cibles(D, parti, seuil=th))
        b = mesure.bilan(D, V, f"{nom} · θ={th}")
        L.append({"fonds": nom, "θ": th, "excès v1": V1_THETA[nom][th],
                  "excès courante": round(b["excès %/an"], 2),
                  "Δ": round(b["excès %/an"] - V1_THETA[nom][th], 2)})
T = pd.DataFrame(L).set_index(["fonds", "θ"])
display(T)
print("balayage publié en entier — le verdict reste lu à la valeur déclarée d'avance (θ=∞ : M3)")

excès v1  excès courante     Δ
fonds θ                                  
NANC  10       4.71            4.76  0.05
      20       4.62            4.62  0.00
      50       5.14            5.17  0.03
      100      4.75            4.76  0.01
GOP   10       3.02            3.02 -0.00
      20       2.14            2.11 -0.03
      50       1.44            1.45  0.01
      100      1.64            1.64  0.00

balayage publié en entier — le verdict reste lu à la valeur déclarée d'avance (θ=∞ : M3)


In [7]:
from tools.donnees import W3
gam = moteur.gamma_purge(D, seuil=W3)                     # 756 j — la fenêtre du prospectus
V1_NET = {"NANC": 1.31, "GOP": 1.06}
L = []
for parti, nom in [("Democrat", "NANC"), ("Republican", "GOP")]:
    WP = {i: moteur.poids_netting(D, gam, i, parti) for i in D.coupes[:-1]}
    V, _ = moteur.run_livre(D, WP)
    b = mesure.bilan(D, V, f"{nom} · netting (courante)")
    L.append({"fonds": nom, "M3 courante": round(B[nom]["excès %/an"], 2),
              "avec netting v1": V1_NET[nom], "avec netting courante": round(b["excès %/an"], 2)})
display(pd.DataFrame(L).set_index("fonds"))
print("le netting reste NON retenu (règle des deux camps) — mesuré ici pour la continuité")

,M3 courante,avec netting v1,avec netting courante
fonds,,,
NANC,3.42,1.31,1.28
GOP,1.21,1.06,1.03


le netting reste NON retenu (règle des deux camps) — mesuré ici pour la continuité


## §5 — La version ETF, carte sectorielle datée

In [8]:
CARTE, carte_hist = etf.carte_datee(D)
V1_ETF = {"NANC": (1.78, 1.66), "GOP": (0.78, 1.02), "UNIQUE": (1.45, 2.05)}
NAV_ETF = {}
L = []
for parti, nom in [("Democrat", "NANC"), ("Republican", "GOP"), (None, "UNIQUE")]:
    NAV_ETF[nom], _ = moteur.run_livre(D, etf.cibles_etf(D, parti, carte_hist=carte_hist))
    b = mesure.bilan_ic(D, NAV_ETF[nom], f"ETF daté · {nom} (courante)")
    L.append({"signal": nom, "excès v1": V1_ETF[nom][0], "excès courante": round(b["excès %/an"], 2),
              "t v1": V1_ETF[nom][1], "t courante": round(b["t"], 2), "NAV": round(b["NAV"], 2)})
display(pd.DataFrame(L).set_index("signal"))

,excès v1,excès courante,t v1,t courante,NAV
signal,,,,,
NANC,1.78,1.80,1.66,1.67,587.40
GOP,0.78,0.80,1.02,1.07,553.29
UNIQUE,1.45,1.49,2.05,2.08,578.78


## §6 — Les deux poches (le livrable du §20), sur la table courante

In [9]:
poches.ajouter_spy(D)
WS = etf.cibles_etf(D, None, carte_hist=carte_hist)       # la poche active = l'unique, carte datée
V_SIG, d = poches.serie_active(D, WS)
assert abs(V_SIG.iloc[-1] - NAV_ETF["UNIQUE"].iloc[-1]) < 1e-9
print(f"poche active = l'unique carte datée — NAV {V_SIG.iloc[-1]:.2f} · d : {len(d):,} jours "
      f"({d.index[0].date()} → {d.index[-1].date()})")

# le profil statique : l'IR doit être CONSTANT (l'identité du §20.3)
irs = {}
for a in [0.25, 0.5, 1.0]:
    WP = poches.livre_deux_poches(D, lambda i, a=a: a, WS)
    V, _ = moteur.run_livre(D, WP)
    m = poches.mesures(D, V, f"poches a={a}")
    irs[a] = m["IR"]
irs = pd.Series(irs)
assert irs.max()/irs.min() - 1 < 0.01, f"IR non constant : {irs.to_dict()}"
print(f"✅ IR constant à travers les doses : {irs.min():.4f} → {irs.max():.4f} "
      f"(v1 publié : 0,637 · l'identité tient, la valeur peut bouger avec la table)")

poche active = l'unique carte datée — NAV 578.78 · d : 3,101 jours (2014-03-04 → 2026-07-01)
✅ IR constant à travers les doses : 0.6477 → 0.6482 (v1 publié : 0,637 · l'identité tient, la valeur peut bouger avec la table)


In [10]:
SIG_AT = poches.sigma_roll(D, d, H=W3)
CUTS_P = [i for i in D.coupes if np.isfinite(SIG_AT.iloc[i])]
L = []
for te in [1.0, 2.0, 3.0]:
    A_ = poches.doses(D, SIG_AT, te, bande=0.20, cuts=CUTS_P)
    WP = poches.livre_deux_poches(D, lambda i, A_=A_: A_.get(i), WS, cuts=CUTS_P)
    V, _ = moteur.run_livre(D, WP, cuts=CUTS_P)
    m = poches.mesures(D, V, f"pilote TE*={te}", WP=WP, cuts=CUTS_P)
    L.append({"TE* %": te, "a médian": round(float(np.median(list(A_.values()))), 3),
              "coupes à a=1": sum(1 for v in A_.values() if v >= 1 - 1e-12),
              "TE réalisée %": round(m["TE %/an"], 2), "excès net %/an": round(m["excès NET %/an"], 2),
              "IR": round(m["IR"], 3), "NAV": round(m["NAV"], 1)})
display(pd.DataFrame(L).set_index("TE* %"))

cal_ = poches.calibration(D, d, SIG_AT, H=W3, cuts=CUTS_P)
print(f"calibration (TE réalisée ÷ prédite) : médiane {cal_['médiane']:.3f} (v1 : 1,257) · "
      f"dépassements {cal_['part > 1']:.0%} (v1 : 61 %) · q95 {cal_['q95']:.2f}")

i_last = CUTS_P[-2]
s_hat = SIG_AT.iloc[i_last]; a_last = min(1.0, 2.0/s_hat)
w_last = pd.Series(poches.livre_deux_poches(D, lambda i: a_last, WS, cuts=CUTS_P)[i_last]).sort_values(ascending=False)
print(f"\ndernière coupe ({D.cal[i_last].date()}, TE*=2 %) : σ̂ {s_hat:.2f} % ⇒ a = {a_last:.3f} "
      f"(v1 : 0,832) — {len(w_last)} lignes, SPY {100*w_last.get('SPY', 0):.1f} %")
print("   " + " · ".join(f"{t} {100*v:.1f}%" for t, v in w_last.head(6).items()))

,a médian,coupes à a=1,TE réalisée %,excès net %/an,IR,NAV
TE* %,,,,,,
1.0,0.429,0,1.19,0.81,0.694,391.9
2.0,1.000,64,2.09,1.37,0.658,409.3
3.0,1.000,111,2.34,1.59,0.669,415.8


calibration (TE réalisée ÷ prédite) : médiane 1.276 (v1 : 1,257) · dépassements 60% (v1 : 61 %) · q95 1.78

dernière coupe (2026-05-29, TE*=2 %) : σ̂ 2.42 % ⇒ a = 0.826 (v1 : 0,832) — 12 lignes, SPY 17.4 %
   XLK 38.4% · SPY 17.4% · XLC 11.9% · XLV 9.4% · XLY 8.0% · XLF 6.2%


## §7 — La vague, en un tableau : ce qui bouge de v1 à la table courante

In [11]:
import json as _json
ANCRES = {
  "flux": len(D.df),
  "M3": {n: {"excès": round(B[n]["excès %/an"], 2), "t": round(B[n]["t"], 2),
             "NAV": round(B[n]["NAV"], 2)} for n in ["NANC", "GOP", "UNIQUE"]},
  "ETF_unique": {"excès": round(mesure.bilan(D, NAV_ETF["UNIQUE"], "etf u")["excès %/an"], 2),
                 "NAV": round(NAV_ETF["UNIQUE"].iloc[-1], 2)},
  "IR_poches": round(float(irs.mean()), 4),
  "calibration_mediane": round(cal_["médiane"], 3),
}
print(_json.dumps(ANCRES, indent=2, ensure_ascii=False))
with open("ancres_table_courante.json", "w") as f:
    _json.dump(ANCRES, f, indent=2, ensure_ascii=False)
print("\n→ ancres de la table courante écrites dans ancres_table_courante.json")
print("   (tools/test_ancres.py asserte les ancres v1 sur la table v1 ; ces valeurs-ci sont")
print("    les ancres de la génération 2 — toute dérive future se lira contre ce fichier)")

{
  "flux": 113645,
  "M3": {
    "NANC": {
      "excès": 3.42,
      "t": 1.61,
      "NAV": 662.37
    },
    "GOP": {
      "excès": 1.21,
      "t": 1.59,
      "NAV": 572.1
    },
    "UNIQUE": {
      "excès": 2.51,
      "t": 1.92,
      "NAV": 628.75
    }
  },
  "ETF_unique": {
    "excès": 1.49,
    "NAV": 578.78
  },
  "IR_poches": 0.648,
  "calibration_mediane": 1.276
}

→ ancres de la table courante écrites dans ancres_table_courante.json
   (tools/test_ancres.py asserte les ancres v1 sur la table v1 ; ces valeurs-ci sont
    les ancres de la génération 2 — toute dérive future se lira contre ce fichier)


## Lecture

Les corrections de la table (NOTE_DIFF : tickers de parenthèse récupérés, suffixes de classe,
FISV, sous-commissions…) déplacent les résultats **de quelques centièmes de point** — dans le
bruit des intervalles de confiance publiés, et **aucune conclusion ne change** : le produit
dépasse ses composantes, aucun excès ne franchit son seuil, l'IR des deux poches est constant,
et TE\* reste la seule question ouverte (celle du client).

`FICHE_M3` et la carte restent adossées à la table v1 archivée — ce notebook est la passerelle
chiffrée entre les deux mondes, et le point de départ de toute strate future (qui travaillera,
elle, sur la table courante, avec `tools/`).